# Tenfold Tuning Runner

This notebook loads the setup cells from `ten_min_segment_pipeline_v10.4.ipynb`, then runs a grouped 10-fold class-weight x label-smoothing sweep with pooled out-of-fold Severe calibration.

The sweep is executed sequentially with aggressive TensorFlow and plot cleanup between folds and between weight settings to reduce CPU memory buildup.

In [ ]:
from pathlib import Path
import os
import runpy
import sys


def resolve_project_root() -> Path:
    env_root = os.environ.get("CTG_DISSERTATION_ROOT")
    candidate_starts = [Path.cwd().resolve()]
    if env_root:
        candidate_starts.append(Path(env_root).expanduser().resolve())

    candidate_starts.extend(
        [
            Path("/content/CTG_Dissertation"),
            Path("/content/drive/MyDrive/CTG_Dissertation"),
        ]
    )

    seen = set()
    for start in candidate_starts:
        if start in seen:
            continue
        seen.add(start)

        for candidate in [start, *start.parents]:
            runner_candidate = candidate / "src" / "Transfer_Learning" / "run_tenfold_tuning_from_notebook.py"
            if runner_candidate.exists():
                return candidate

    raise FileNotFoundError(
        "Could not locate the CTG_Dissertation project root. "
        "In Colab, clone or copy the repo to /content/CTG_Dissertation, "
        "or set CTG_DISSERTATION_ROOT before running this cell."
    )


PROJECT_ROOT = resolve_project_root()
TRANSFER_LEARNING_DIR = PROJECT_ROOT / "src" / "Transfer_Learning"
os.chdir(TRANSFER_LEARNING_DIR)

if str(TRANSFER_LEARNING_DIR) not in sys.path:
    sys.path.insert(0, str(TRANSFER_LEARNING_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {TRANSFER_LEARNING_DIR}")

runner_path = TRANSFER_LEARNING_DIR / "run_tenfold_tuning_from_notebook.py"
runner_globals = runpy.run_path(str(runner_path))
run_result = runner_globals["run"]()

tenfold_tuning_summary_df = run_result["tenfold_tuning_summary_df"]
tenfold_tuning_summaries = run_result["tenfold_tuning_summaries"]
best_tenfold_run = run_result["best_tenfold_run"]
best_current_weights = run_result["best_current_weights"]
tenfold_tuning_output_dir = run_result["tenfold_tuning_output_dir"]

tenfold_tuning_summary_df

FileNotFoundError: [Errno 2] No such file or directory: '/content/run_tenfold_tuning_from_notebook.py'

In [ ]:
print('Best setting:')
print(f"  Manual class weights: {best_tenfold_run['manual_class_weights']}")
print(f"  Label smoothing: {best_tenfold_run['label_smoothing']:.2f}")
print(f"  OOF Severe boost: {best_tenfold_run['best_oof_severe_boost']:.2f}")
print(f"  Raw balanced accuracy: {best_tenfold_run['raw_balanced_accuracy']:.4f}")
print(f"  Calibrated balanced accuracy: {best_tenfold_run['calibrated_balanced_accuracy']:.4f}")
print(f"  Raw macro F1: {best_tenfold_run['raw_macro_f1']:.4f}")
print(f"  Calibrated macro F1: {best_tenfold_run['calibrated_macro_f1']:.4f}")
print(f"  Outputs: {tenfold_tuning_output_dir}")